# Entraînement des modèles

**Objectif du notebook:**
Ce notebook est la suite du notebook *2.Data cleaning et processing*. Il vise entrainer un modèle standard qui servira de baseline (référence), puis de comparer différents modèles, avec différents paramètres pour obtenir la meilleure valeur de RMSE.

---

# 0. Imports

In [21]:
import pandas as pd
import numpy as np

from scipy import sparse

from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import Ridge, ElasticNet

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

# 1. Chargement des données

In [22]:
X_train = sparse.load_npz("../data/X_train.npz")
X_test = sparse.load_npz("../data/X_test.npz")

y_train = pd.read_csv("../data/y_train.csv").squeeze()

sample_submission = pd.read_csv(
    "../data/sample_submission.csv"
)

print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("y_train :", y_train.shape)

X_train : (54000, 2018)
X_test : (36000, 2018)
y_train : (54000,)


In [23]:
y_train_log = np.log1p(y_train)

print(y_train.describe())

count    5.400000e+04
mean     1.100337e+04
std      3.339099e+04
min      1.218868e+02
25%      9.263384e+02
50%      3.371242e+03
75%      8.197249e+03
max      4.027136e+06
Name: UltimateIncurredClaimCost, dtype: float64


# 2. Baseline

In [24]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

baseline_scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train)):

    X_tr = X_train[train_idx]
    X_val = X_train[valid_idx]

    y_tr = y_train_log.iloc[train_idx]
    y_val = y_train_log.iloc[valid_idx]

    model = Ridge(alpha=1.0)

    model.fit(X_tr, y_tr)

    pred_log = model.predict(X_val)

    pred_log = np.clip(pred_log, 0, 20)
    
    pred = np.expm1(pred_log)
    y_true = np.expm1(y_val)

    rmse = root_mean_squared_error(y_true, pred)

    baseline_scores.append(rmse)

    print(f"Fold {fold+1} RMSE : {rmse:.2f}")

print("\nBaseline RMSE moyen :", np.mean(baseline_scores))

Fold 1 RMSE : 9724942.75
Fold 2 RMSE : 8563221.66
Fold 3 RMSE : 6601842.38
Fold 4 RMSE : 7973549.27
Fold 5 RMSE : 8337292.68

Baseline RMSE moyen : 8240169.748303363


# 3. Entrainement du modèle

In [25]:
import warnings
warnings.filterwarnings("ignore")

results = []

models = {

    "Ridge_1": Ridge(alpha=1.0),

    "Ridge_10": Ridge(alpha=10.0),

    "ElasticNet_1": ElasticNet(
        alpha=0.001,
        l1_ratio=0.5,
        max_iter=5000
    ),

    "LightGBM_small": LGBMRegressor(
        objective="regression",
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM_large": LGBMRegressor(
        objective="regression",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost_small": XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost_large": XGBRegressor(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


for model_name, model in models.items():

    print("=" * 60)
    print(f"Training : {model_name}")

    fold_rmse = []

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train)):

        X_tr = X_train[train_idx]
        X_val = X_train[valid_idx]

        y_tr = y_train_log.iloc[train_idx]
        y_val = y_train_log.iloc[valid_idx]

        # =========================
        # TRAIN
        # =========================

        model.fit(X_tr, y_tr)

        # =========================
        # PREDICT
        # =========================

        pred_log = model.predict(X_val)

        pred_log = np.clip(pred_log, 0, 20)

        pred = np.expm1(pred_log)

        y_true = np.expm1(y_val)

        # =========================
        # RMSE
        # =========================

        rmse = root_mean_squared_error(y_true, pred)

        fold_rmse.append(rmse)

        print(f"Fold {fold+1} RMSE : {rmse:.2f}")

    # =========================
    # GLOBAL SCORE
    # =========================

    mean_rmse = np.mean(fold_rmse)
    std_rmse = np.std(fold_rmse)

    print(f"\nMean RMSE : {mean_rmse:.2f}")
    print(f"Std RMSE : {std_rmse:.2f}")

    results.append({
        "Model": model_name,
        "Mean_RMSE": mean_rmse,
        "Std_RMSE": std_rmse
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Mean_RMSE"
)

results_df.reset_index(drop=True, inplace=True)

results_df

Training : Ridge_1
Fold 1 RMSE : 9724942.75
Fold 2 RMSE : 8563221.66
Fold 3 RMSE : 6601842.38
Fold 4 RMSE : 7973549.27
Fold 5 RMSE : 8337292.68

Mean RMSE : 8240169.75
Std RMSE : 1007328.08
Training : Ridge_10
Fold 1 RMSE : 9724943.22
Fold 2 RMSE : 8563221.91
Fold 3 RMSE : 6601842.39
Fold 4 RMSE : 7973549.88
Fold 5 RMSE : 8337295.33

Mean RMSE : 8240170.55
Std RMSE : 1007328.25
Training : ElasticNet_1
Fold 1 RMSE : 6909206.24
Fold 2 RMSE : 6616791.83
Fold 3 RMSE : 4689315.07
Fold 4 RMSE : 4884148.04
Fold 5 RMSE : 8089665.50

Mean RMSE : 6237825.34
Std RMSE : 1284835.10
Training : LightGBM_small
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.068215 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 75240
[LightGBM] [Info] Number of data points in the train set: 43200, number of used features: 1999
[LightGBM] [Info] Start training from score 8.066852
Fold 1 RMSE : 25427.95
[LightGBM] [Info] Auto-choosin

,Model,Mean_RMSE,Std_RMSE
0,LightGBM_small,2.809444e+04,8.176100e+03
1,XGBoost_large,2.810922e+04,8.203536e+03
2,XGBoost_small,2.812065e+04,8.214489e+03
3,LightGBM_large,2.818433e+04,8.181194e+03
4,ElasticNet_1,6.237825e+06,1.284835e+06
5,Ridge_1,8.240170e+06,1.007328e+06
6,Ridge_10,8.240171e+06,1.007328e+06


In [26]:
best_model = LGBMRegressor(
    objective="regression",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train_log)

test_pred_log = best_model.predict(X_test)

test_pred_log = np.clip(
    test_pred_log,
    0,
    np.log1p(y_train.max())
)

test_pred = np.expm1(test_pred_log)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.082410 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 85538
[LightGBM] [Info] Number of data points in the train set: 54000, number of used features: 2018
[LightGBM] [Info] Start training from score 8.064895


In [29]:
submission = sample_submission.copy()

submission["UltimateIncurredClaimCost"] = test_pred

submission.head()

,ClaimNumber,UltimateIncurredClaimCost
0,WC8145235,4928.815507
1,WC2005111,3089.444914
2,WC6899143,18057.172283
3,WC5502023,293.175641
4,WC4785156,2030.290943


In [30]:
submission.to_csv(
    "../data/lgb_submission.csv",
    index=False
)

Point d'amélioration: 
- Optimisation des hyperparamètres (GridSearch)
- Ajout d’early stopping (éviter l'overfitting)
- Utilisation de modèles NLP avancés pour les descriptions de sinistres